In [ ]:
from pathlib import Path
parent_directory = Path.cwd().parent.parent
print(parent_directory) 
type(parent_directory)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
#reemplazar con la ruta de los archivos correcta
parent_directory = Path.cwd().parent.parent
FCST=pd.read_excel(parent_directory / "FCST Merck Abril 26.xlsx",sheet_name="Abril 2026")
PROD_DC_2=pd.read_excel(parent_directory / "PRODUCTOS DC.xlsx",sheet_name="Catalogo")

In [ ]:
"""""
Module: P&G Forecast extraction 2
Purpose: Extraer ## de jeringas para todos los meses por producto
Date: 30/07/2026
Author: J.Gonzalez
"""
id_cols = ['SKUMERCK']
#Calculo de mes y año actual
Month_today=pd.to_datetime("today").month
Year_today=pd.to_datetime("today").year
demand_today=pd.to_datetime(str(Year_today) + "-01-" + str(Month_today), format="%Y-%m-%d")
demand_today=str(demand_today)[0:10]

#seleccion de meses con demanda futura en el FCST
matching_cols = []
for i in range(len(FCST.columns) - 1, -1, -1):  # from [-1] backwards
    col = FCST.columns[i]
    parsed = pd.to_datetime(col, errors='coerce')
    if pd.notna(parsed):
        matching_cols.append(col)
        if parsed.month <= Month_today and parsed.year <= Year_today:
            break  #
matching_cols = list(reversed(matching_cols))

#Union de columnas id con columna demanda de cada mes
FCST_month = FCST[id_cols + matching_cols].copy()
FCST_month = FCST_month.dropna(subset=['SKUMERCK']).reset_index(drop=True)

In [ ]:
"""""
Module: Produccion por linea 2
Purpose: Separa los productos por familia, y despues por linea que utiliza dentro de cada mes esa familia especifica
Date: 30/07/2026
Author: J.Gonzalez
"""
#Merge de datos y eliminacion de SKU
PROD_DC_2 = PROD_DC_2.merge(FCST_month, on='SKUMERCK', how='left')
PROD_DC_2.drop(['SKUMERCK'], axis=1, inplace=True)

#dicccionario para la suma del group
PROD_DC_2.columns = [pd.to_datetime(col, errors='coerce').strftime('%m-%Y') if pd.notna(pd.to_datetime(col, errors='coerce')) else col for col in PROD_DC_2.columns]
agg_dict = {col: 'sum' for col in PROD_DC_2.columns if col not in ['Nombre granel', 'Linea Granel 1', 'Linea Granel 2']}

#Group by de familias de granel
PROD_DC_2 = PROD_DC_2.groupby(['Nombre granel', 'Linea Granel 1', 'Linea Granel 2'], as_index=False).agg(agg_dict)

#Se reduce de dos columnas a una para identificar DC
PROD_DC_2['Linea Granel 1'] = np.where(
    (PROD_DC_2['Linea Granel 1'] == 1) & (PROD_DC_2['Linea Granel 2'] == 1),
    'DC2',
    'DC1')
PROD_DC_2.drop(['Linea Granel 2'], axis=1, inplace=True)
PROD_DC_2 = PROD_DC_2.rename(columns={'Linea Granel 1': 'DC'})



In [ ]:
"""""
Module: Balanceo mensual
Purpose: Balancea DC2 mandando su residuo a DC1 para cada familia en cada mes 
Date: 30/07/2026
Author: J.Gonzalez
"""
# Numero de jeringas por lote y redondeo de residuo
jeringas=115200
decimales_req=2

# Identifica todas las familias disponibles y registra cuales tienen capacidad en DC2
familias= PROD_DC_2['Nombre granel'].unique()
familias_con_dc2=PROD_DC_2.loc[PROD_DC_2['DC'] == 'DC2', 'Nombre granel'].unique()

#Base de datos para residuos en jeringas
registro_residuos = PROD_DC_2.copy()
# Balanceo, en casos que detecte que la familia con DC2 tiene residuo mando todo ese residuo a DC1 de esa misma familia                     

for familia in familias_con_dc2:
    for col in agg_dict.keys():
        # Filtra las filas para esta familia específica a DC2 y DC1
        mask_dc2 = (PROD_DC_2['Nombre granel'] == familia) & (PROD_DC_2['DC'] == 'DC2')
        mask_dc1 = (PROD_DC_2['Nombre granel'] == familia) & (PROD_DC_2['DC'] == 'DC1')
        # Valor de DC2 para la familia y mes especificado
        dc2_val = PROD_DC_2.loc[mask_dc2, col].values[0]

        # Balanceo de residuos DC2
        if dc2_val % jeringas != 0:
            residue = dc2_val % jeringas
            # suma el residuo a DC1
            PROD_DC_2.loc[mask_dc1, col] = (PROD_DC_2.loc[mask_dc1, col] + residue)
            # resta el residuo de DC2
            PROD_DC_2.loc[mask_dc2, col] =(PROD_DC_2.loc[mask_dc2, col] - residue)


for familia in familias:
    for col in agg_dict.keys():
        mask = (PROD_DC_2['Nombre granel'] == familia)
        #calcula el residuo de la familia en dc1
        registro_residuos.loc[mask, col]=jeringas-PROD_DC_2.loc[mask, col]%jeringas
        registro_residuos.loc[mask, col] = np.where(
            (registro_residuos.loc[mask, col] <= jeringas * 0.01) |
            (registro_residuos.loc[mask, col] >= jeringas * (1 - 0.01)),
            0,
            registro_residuos.loc[mask, col])
        #cambia toda la demanda de la tabla de jeringas a lotes
        PROD_DC_2.loc[mask, col] = (PROD_DC_2.loc[mask, col]/jeringas).round(decimales_req)
        
        


# Exportación de resultados en archivos CSV
DC1=PROD_DC_2[PROD_DC_2['DC'] == 'DC1'].copy()
DC2=PROD_DC_2[PROD_DC_2['DC'] == 'DC2'].copy()
DC1.to_csv("DC1_FCST.csv", index=False)
DC2.to_csv("DC2_FCST.csv", index=False)
PROD_DC_2.to_csv("PROD_DC_2_balanceado_2.csv", index=False)
registro_residuos.to_csv("registro_residuos.csv", index=False)

